<a href="https://colab.research.google.com/github/seethaladevi2024-cpu/OIBSIP/blob/main/Calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Calculator Web Application for Google Colab
# A simple calculator that performs basic arithmetic operations

# Step 1: Install required packages
print("📦 Installing required packages...")
!pip install flask flask-cors -q

print("✅ Packages installed successfully!\n")

# Step 2: Import necessary libraries
from flask import Flask, render_template_string, request, jsonify
from flask_cors import CORS
import threading
from google.colab.output import eval_js

# Step 3: Initialize Flask application
app = Flask(__name__)
CORS(app)  # Enable CORS for cross-origin requests

# Step 4: Define the HTML template with embedded CSS and JavaScript
HTML_TEMPLATE = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Calculator App</title>
    <style>
        /* Reset and base styles */
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
            padding: 20px;
        }

        /* Main container */
        .container {
            background: white;
            border-radius: 25px;
            box-shadow: 0 25px 70px rgba(0, 0, 0, 0.3);
            padding: 40px;
            max-width: 450px;
            width: 100%;
        }

        /* Header */
        .header {
            text-align: center;
            margin-bottom: 30px;
        }

        .icon {
            font-size: 60px;
            margin-bottom: 10px;
        }

        h1 {
            color: #333;
            font-size: 32px;
            margin-bottom: 5px;
        }

        .subtitle {
            color: #666;
            font-size: 14px;
        }

        /* Calculator display */
        .display {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 25px;
            border-radius: 15px;
            margin-bottom: 25px;
            box-shadow: inset 0 4px 8px rgba(0, 0, 0, 0.2);
        }

        .display-result {
            color: white;
            font-size: 36px;
            font-weight: bold;
            text-align: right;
            min-height: 50px;
            word-wrap: break-word;
            display: flex;
            align-items: center;
            justify-content: flex-end;
        }

        /* Form elements */
        .form-group {
            margin-bottom: 20px;
        }

        label {
            display: block;
            color: #555;
            font-weight: 600;
            margin-bottom: 8px;
            font-size: 14px;
        }

        input[type="number"] {
            width: 100%;
            padding: 15px;
            border: 2px solid #e0e0e0;
            border-radius: 12px;
            font-size: 18px;
            transition: all 0.3s ease;
            background: #f9f9f9;
        }

        input[type="number"]:focus {
            outline: none;
            border-color: #667eea;
            background: white;
            box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.1);
        }

        /* Operation buttons */
        .operations {
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 10px;
            margin-bottom: 20px;
        }

        .operation-btn {
            padding: 15px;
            border: 2px solid #e0e0e0;
            background: white;
            border-radius: 12px;
            font-size: 24px;
            font-weight: 600;
            cursor: pointer;
            transition: all 0.3s ease;
            color: #667eea;
        }

        .operation-btn:hover {
            background: #f0f4ff;
            border-color: #667eea;
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(102, 126, 234, 0.2);
        }

        .operation-btn.active {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            border-color: #667eea;
            color: white;
        }

        /* Calculate button */
        .btn-calculate {
            width: 100%;
            padding: 18px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            border-radius: 12px;
            font-size: 20px;
            font-weight: 600;
            cursor: pointer;
            transition: all 0.3s ease;
            box-shadow: 0 8px 20px rgba(102, 126, 234, 0.3);
        }

        .btn-calculate:hover {
            transform: translateY(-3px);
            box-shadow: 0 12px 30px rgba(102, 126, 234, 0.4);
        }

        .btn-calculate:active {
            transform: translateY(-1px);
        }

        /* Clear button */
        .btn-clear {
            width: 100%;
            padding: 12px;
            background: #e0e0e0;
            color: #666;
            border: none;
            border-radius: 12px;
            font-size: 16px;
            font-weight: 600;
            cursor: pointer;
            transition: all 0.3s ease;
            margin-top: 10px;
        }

        .btn-clear:hover {
            background: #d0d0d0;
        }

        /* Error message */
        .error-message {
            background: #fee;
            color: #c33;
            padding: 12px;
            border-radius: 10px;
            margin-bottom: 15px;
            font-size: 14px;
            text-align: center;
            border: 1px solid #fcc;
            display: none;
        }

        .error-message.show {
            display: block;
        }

        /* Info text */
        .info-text {
            text-align: center;
            color: #999;
            font-size: 13px;
            margin-top: 15px;
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <div class="icon">🔢</div>
            <h1>Calculator</h1>
            <p class="subtitle">Simple and easy to use</p>
        </div>

        <div class="display">
            <div class="display-result" id="result">0</div>
        </div>

        <div class="error-message" id="errorMessage"></div>

        <form id="calculatorForm">
            <div class="form-group">
                <label for="num1">First Number</label>
                <input type="number" id="num1" step="any" placeholder="Enter first number" required>
            </div>

            <div class="form-group">
                <label>Select Operation</label>
                <div class="operations">
                    <button type="button" class="operation-btn active" data-operation="+" title="Addition">+</button>
                    <button type="button" class="operation-btn" data-operation="-" title="Subtraction">−</button>
                    <button type="button" class="operation-btn" data-operation="*" title="Multiplication">×</button>
                    <button type="button" class="operation-btn" data-operation="/" title="Division">÷</button>
                </div>
            </div>

            <div class="form-group">
                <label for="num2">Second Number</label>
                <input type="number" id="num2" step="any" placeholder="Enter second number" required>
            </div>

            <button type="submit" class="btn-calculate">Calculate</button>
            <button type="button" class="btn-clear" onclick="clearCalculator()">Clear</button>
        </form>

        <p class="info-text">Enter two numbers and select an operation</p>
    </div>

    <script>
        let selectedOperation = '+';

        // Handle operation button clicks
        document.querySelectorAll('.operation-btn').forEach(btn => {
            btn.addEventListener('click', function() {
                // Remove active class from all buttons
                document.querySelectorAll('.operation-btn').forEach(b => b.classList.remove('active'));

                // Add active class to clicked button
                this.classList.add('active');

                // Store selected operation
                selectedOperation = this.getAttribute('data-operation');
            });
        });

        // Handle form submission
        document.getElementById('calculatorForm').addEventListener('submit', async function(e) {
            e.preventDefault();

            const num1 = parseFloat(document.getElementById('num1').value);
            const num2 = parseFloat(document.getElementById('num2').value);
            const errorMessage = document.getElementById('errorMessage');
            const resultDisplay = document.getElementById('result');

            // Hide previous error
            errorMessage.classList.remove('show');

            // Validate inputs
            if (isNaN(num1) || isNaN(num2)) {
                showError('Please enter valid numbers');
                return;
            }

            // Show loading state
            resultDisplay.textContent = '...';

            try {
                // Send calculation request to backend
                const response = await fetch('/calculate', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({
                        num1: num1,
                        num2: num2,
                        operation: selectedOperation
                    })
                });

                const data = await response.json();

                if (data.error) {
                    showError(data.error);
                    resultDisplay.textContent = '0';
                } else {
                    // Display result with animation
                    resultDisplay.textContent = data.result;
                }
            } catch (error) {
                showError('An error occurred. Please try again.');
                resultDisplay.textContent = '0';
            }
        });

        // Function to show error message
        function showError(message) {
            const errorMessage = document.getElementById('errorMessage');
            errorMessage.textContent = '⚠️ ' + message;
            errorMessage.classList.add('show');
        }

        // Function to clear calculator
        function clearCalculator() {
            document.getElementById('num1').value = '';
            document.getElementById('num2').value = '';
            document.getElementById('result').textContent = '0';
            document.getElementById('errorMessage').classList.remove('show');

            // Reset to addition
            document.querySelectorAll('.operation-btn').forEach(b => b.classList.remove('active'));
            document.querySelector('.operation-btn[data-operation="+"]').classList.add('active');
            selectedOperation = '+';
        }
    </script>
</body>
</html>
'''

# Step 5: Define calculation function
def perform_calculation(num1, num2, operation):
    """
    Perform arithmetic calculation based on the operation

    Args:
        num1: First number
        num2: Second number
        operation: Operation to perform (+, -, *, /)

    Returns:
        Tuple of (result, error_message)
    """
    try:
        # Convert to float for calculations
        num1 = float(num1)
        num2 = float(num2)

        # Perform the operation
        if operation == '+':
            result = num1 + num2
        elif operation == '-':
            result = num1 - num2
        elif operation == '*':
            result = num1 * num2
        elif operation == '/':
            # Check for division by zero
            if num2 == 0:
                return None, "Cannot divide by zero!"
            result = num1 / num2
        else:
            return None, "Invalid operation!"

        # Format result - remove unnecessary decimals
        if result == int(result):
            result = int(result)
        else:
            result = round(result, 10)  # Round to 10 decimal places

        return result, None

    except ValueError:
        return None, "Invalid input! Please enter valid numbers."
    except Exception as e:
        return None, f"An error occurred: {str(e)}"

# Step 6: Define routes

@app.route('/')
def home():
    """Render the calculator page"""
    return render_template_string(HTML_TEMPLATE)

@app.route('/calculate', methods=['POST'])
def calculate():
    """
    Handle calculation requests from the frontend
    Receives two numbers and an operation, returns the result
    """
    try:
        # Get JSON data from request
        data = request.get_json()
        num1 = data.get('num1')
        num2 = data.get('num2')
        operation = data.get('operation')

        # Validate inputs
        if num1 is None or num2 is None or operation is None:
            return jsonify({'error': 'Missing required parameters'}), 400

        # Perform calculation
        result, error = perform_calculation(num1, num2, operation)

        if error:
            return jsonify({'error': error}), 400

        return jsonify({'result': result})

    except Exception as e:
        return jsonify({'error': 'An error occurred during calculation'}), 500

# Step 7: Run the application
def main():
    """Main function to start the Flask application"""
    print("🚀 Starting Flask server...")

    # Start Flask in a background thread
    threading.Thread(
        target=lambda: app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False),
        daemon=True
    ).start()

    # Wait for Flask to start
    import time
    time.sleep(3)

    # Get public URL from Colab
    print("🌐 Getting public URL from Google Colab...")
    try:
        public_url = eval_js("google.colab.kernel.proxyPort(5000)")
        print(f"\n" + "="*60)
        print(f"✅ SUCCESS! Your Calculator App is running!")
        print(f"="*60)
        print(f"\n🔗 Public URL: {public_url}")
        print(f"\n📱 Click the link above to access your calculator!")
        print(f"\n⚠️  Keep this cell running to maintain the connection.")
        print(f"\n🔢 Features:")
        print(f"   • Addition (+)")
        print(f"   • Subtraction (−)")
        print(f"   • Multiplication (×)")
        print(f"   • Division (÷)")
        print(f"   • Division by zero protection")
        print(f"   • Input validation")
        print(f"\n💡 Tip: Enter numbers and select an operation to calculate!")
        print(f"="*60)
    except Exception as e:
        print(f"\n⚠️  Could not automatically get public URL.")
        print(f"📋 The Flask server is running on port 5000")
        print(f"   Look for the 'Open in new tab' icon in the cell output")

# Execute the main function
main()

📦 Installing required packages...
✅ Packages installed successfully!

🚀 Starting Flask server...
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


🌐 Getting public URL from Google Colab...

✅ SUCCESS! Your Calculator App is running!

🔗 Public URL: https://5000-m-s-2bd8v4xbehndg-a.asia-east1-0.prod.colab.dev

📱 Click the link above to access your calculator!

⚠️  Keep this cell running to maintain the connection.

🔢 Features:
   • Addition (+)
   • Subtraction (−)
   • Multiplication (×)
   • Division (÷)
   • Division by zero protection
   • Input validation

💡 Tip: Enter numbers and select an operation to calculate!
